# Evaluator Module
The Evaluator module creates evaluation reports.

Reports contain evaluation metrics depending on models specified in the evaluation config.

In [2]:
# reloads modules automatically before entering the execution of code
%load_ext autoreload
%autoreload 2

# third parties imports
import numpy as np 
import pandas as pd
# -- add new imports here --

# local imports
from configs import EvalConfig
from constants import Constant as C
from loaders import export_evaluation_report
from loaders import load_ratings
# -- add new imports here --
from surprise import accuracy
from surprise.model_selection import cross_validate, train_test_split, LeaveOneOut
from models import get_top_n, get_top_n_reranked
import random as rd

# 1. Try the loader with surprise_format set to True
data = load_ratings(surprise_format=True)

# 2. Verify the results
print(f"Object Type: {type(data)}")

# 3. Build a trainset to confirm data is correctly loaded (check)
trainset = data.build_full_trainset()
print(f"Number of ratings: {trainset.n_ratings}")
print(f"Number of users: {trainset.n_users}")
print(f"Number of items: {trainset.n_items}")

print("Loader successfully tested in Surprise format!")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Object Type: <class 'surprise.dataset.DatasetAutoFolds'>
Number of ratings: 381181
Number of users: 1000
Number of items: 8737
Loader successfully tested in Surprise format!


# 1. Model validation functions
Validation functions are a way to perform crossvalidation on recommender system models. 

In [3]:
def generate_split_predictions(algo, ratings_dataset, eval_config):
    """Generate predictions on a random test set specified in eval_config"""
    # Split the dataset into train and test sets using the size from eval_config
    trainset, testset = train_test_split(
        ratings_dataset, 
        test_size=eval_config.test_size, 
        random_state=42
    )
    
    # Train the algorithm on the training set
    algo.fit(trainset)
    
    # Generate predictions on the test set
    predictions = algo.test(testset)
    return predictions

def generate_loo_top_n(algo, ratings_dataset, eval_config):
    """Generate top-n recommendations for each user on a random Leave-one-out split (LOO)"""
    # Initialize the LeaveOneOut cross-validator
    loo = LeaveOneOut(n_splits=1, random_state=1)
    
    # Iterate through the split (only one iteration for n_splits=1)
    for trainset, testset in loo.split(ratings_dataset):
        # Train the algorithm on the training set
        algo.fit(trainset)
        
        # Build the anti-testset (all pairs of user-item NOT in the training set)
        anti_testset = trainset.build_anti_testset()
         #J'ai ajouté ce bloc pour accelérer le temps de calcul
        # import random
        # MAX_ANTI_TESTSET = 500_000

        # if len(anti_testset) > MAX_ANTI_TESTSET:
        #     random.seed(0)
        #     anti_testset = random.sample(anti_testset, MAX_ANTI_TESTSET)
        # # Generate predictions for the anti-testset
        predictions = algo.test(anti_testset)
        
        # Get top-n recommendations using the value from eval_config
        # We assign it to anti_testset_top_n as per the return requirement
        anti_testset_top_n = get_top_n(predictions, n=eval_config.top_n_value)
        
    return anti_testset_top_n, testset


def generate_full_top_n(algo, ratings_dataset, eval_config, precomputed_dict=None):
    """Generate top-n recommendations for each user with full training set (LOO)"""
    # Build a training set using 100% of the available data
    full_trainset = ratings_dataset.build_full_trainset()
    
    # Train the algorithm on the full dataset
    algo.fit(full_trainset)
    
    # Build the anti-testset (all user-item pairs not present in the training data)
    anti_testset = full_trainset.build_anti_testset()
    
    # J'ai ajouté ce bloc pour accelérer le temps de calcul
    # import random
    # MAX_ANTI_TESTSET = 500_000
    # if len(anti_testset) > MAX_ANTI_TESTSET:
    #     random.seed(0)
    #     anti_testset = random.sample(anti_testset, MAX_ANTI_TESTSET)
        
    # Generate predictions for the items users haven't seen yet
    predictions = algo.test(anti_testset)
    
    # AJOUT : Si on a les fréquences d'items, on applique le re-ranking
    if precomputed_dict and "item_freq" in precomputed_dict:
        item_freq = precomputed_dict["item_freq"]
        return get_top_n_reranked(predictions, item_freq, alpha=0.1, n=eval_config.top_n_value)
        
    # Retour par défaut si precomputed_dict n'est pas fourni
    return get_top_n(predictions, n=eval_config.top_n_value)


#########################################
# Test of the three functions as required
#########################################
from models import ModelBaseline1
# from evaluation import generate_split_predictions, generate_loo_top_n, generate_full_top_n

# 1. Initialize the configuration instance

eval_config = EvalConfig()

# 2. Load the dataset in Surprise format
data = load_ratings(surprise_format=True)

# 3. Initialize the algorithm
algo = ModelBaseline1()

# --- (a) Test: Simple Train-Test Split ---
# Pass the instance 'eval_config' instead of the class 'EvalConfig'
predictions_a = generate_split_predictions(algo, data, eval_config)
print(f"Test (a): Success. {len(predictions_a)} raw predictions generated.")

# --- (b) Test: Leave-One-Out + Top-N ---
top_n_b, testset_b = generate_loo_top_n(algo, data, eval_config)
print(f"Test (b): Success. {len(top_n_b)} users with recommendations.")
for uid, recs in list(top_n_b.items())[:3]: 
    print(f"  User {uid}: {len(recs)} recommendations -> {recs}")

# --- (c) Test: Full Dataset Training + Top-N ---
top_n_c = generate_full_top_n(algo, data, eval_config)
print(f"Test (c): Success. {len(top_n_c)} users with recommendations.")
for uid, recs in list(top_n_c.items())[:3]:  
    print(f"  User {uid}: {len(recs)} recommendations -> {recs}")

# End of the test

def precompute_information(df_ratings):
    """Returns a dictionary of precomputed information used by full-mode metrics.

    Keys:
    - item_to_rank   : {movie_id: popularity rank}  — used by novelty (rang)
    - n_users        : total number of unique users  — used by MIUF
    - item_freq      : {movie_id: nb users who rated it} — used by MIUF
    - genre_vectors  : {movie_id: normalized genre vector} — used by ILD
    - user_history   : {user_id: set of movie_ids rated by user} — used by serendipity
    """
    from loaders import load_items
    precomputed_dict = {}

    # item_to_rank (novelty rang)
    movie_counts = df_ratings[C.ITEM_ID_COL].value_counts()
    precomputed_dict["item_to_rank"] = movie_counts.rank(ascending=False, method='first').to_dict()

    # n_users + item_freq (MIUF)
    precomputed_dict["n_users"]    = df_ratings[C.USER_ID_COL].nunique()
    precomputed_dict["item_freq"]  = (
        df_ratings.groupby(C.ITEM_ID_COL)[C.USER_ID_COL].nunique().to_dict()
    )

    # genre_vectors (ILD) — build_genre_vectors is defined in the metrics cell below
    df_items = load_items()
    precomputed_dict["genre_vectors"] = build_genre_vectors(df_items)

    # AJOUT : Extraction de l'historique de chaque utilisateur pour get_serendipity()
    precomputed_dict["user_history"] = (
        df_ratings.groupby(C.USER_ID_COL)[C.ITEM_ID_COL]
        .apply(set)
        .to_dict()
    )

    return precomputed_dict           


def create_evaluation_report(eval_config, sp_ratings, precomputed_dict, available_metrics):
    """ Create a DataFrame evaluating various models on metrics specified in an evaluation config.  
    """
    evaluation_dict = {}
    for model_name, model, arguments in eval_config.models:
        print(f'Handling model {model_name}')
        algo = model(**arguments)
        evaluation_dict[model_name] = {}
        
        # Type 1 : split evaluations
        if len(eval_config.split_metrics) > 0:
            print('Training split predictions')
            predictions = generate_split_predictions(algo, sp_ratings, eval_config)
            for metric in eval_config.split_metrics:
                print(f'- computing metric {metric}')
                assert metric in available_metrics['split']
                evaluation_function, parameters =  available_metrics["split"][metric]
                evaluation_dict[model_name][metric] = evaluation_function(predictions, **parameters) 

        # Type 2 : loo evaluations
        if len(eval_config.loo_metrics) > 0:
            print('Training loo predictions')
            
            # 1. On réécrit la logique de génération des prédictions brutes
            from surprise.model_selection import LeaveOneOut
            loo = LeaveOneOut(n_splits=1, random_state=1)
            
            for trainset, testset in loo.split(sp_ratings):
                algo.fit(trainset)
                anti_testset = trainset.build_anti_testset()
                # Génération des scores bruts sans couper au Top-N immédiatement
                raw_predictions = algo.test(anti_testset)
            
            # 2. Extraction du dictionnaire de popularité depuis vos précalculs
            item_freq_distribution = precomputed_dict.get("item_freq", {})
            
            # 3. Application du re-ranking note + popularité
            anti_testset_top_n = get_top_n_reranked(
                raw_predictions, 
                item_freq=item_freq_distribution, 
                alpha=0.1,  # Ajustez cette valeur (ex: 0.05, 0.1, 0.15) pour calibrer le compromis
                n=eval_config.top_n_value
            )
            
            # 4. Calcul des métriques de classement
            for metric in eval_config.loo_metrics:
                assert metric in available_metrics['loo']
                evaluation_function, parameters =  available_metrics["loo"][metric]
                evaluation_dict[model_name][metric] = evaluation_function(anti_testset_top_n, testset, **parameters)
        
        # Type 3 : full evaluations
        if len(eval_config.full_metrics) > 0:
            print('Training full predictions')
            # La correction à appliquer :
            anti_testset_top_n = generate_full_top_n(algo, sp_ratings, eval_config, precomputed_dict)
            for metric in eval_config.full_metrics:
                assert metric in available_metrics['full']
                evaluation_function, parameters =  available_metrics["full"][metric]
                evaluation_dict[model_name][metric] = evaluation_function(
                    anti_testset_top_n,
                    **precomputed_dict,
                    **parameters
                )
        
    return pd.DataFrame.from_dict(evaluation_dict).T

Test (a): Success. 95296 raw predictions generated.
Test (b): Success. 1000 users with recommendations.
  User 277: 80 recommendations -> [(2791, 2), (4284, 2), (4420, 2), (4091, 2), (5040, 2), (60832, 2), (92694, 2), (3805, 2), (72, 2), (127202, 2), (83, 2), (96911, 2), (4130, 2), (468, 2), (5682, 2), (176, 2), (7228, 2), (2282, 2), (3197, 2), (2022, 2), (43921, 2), (1532, 2), (4932, 2), (48516, 2), (32289, 2), (8511, 2), (2581, 2), (4621, 2), (32139, 2), (4713, 2), (3543, 2), (8133, 2), (25828, 2), (6663, 2), (94, 2), (1904, 2), (41527, 2), (690, 2), (4186, 2), (4964, 2), (110781, 2), (2606, 2), (7196, 2), (124859, 2), (5237, 2), (26974, 2), (2769, 2), (8273, 2), (5989, 2), (32892, 2), (7437, 2), (115170, 2), (54419, 2), (7782, 2), (4681, 2), (26686, 2), (32153, 2), (5901, 2), (1707, 2), (64285, 2), (58295, 2), (8836, 2), (2191, 2), (4744, 2), (5899, 2), (714, 2), (6096, 2), (129514, 2), (3987, 2), (2276, 2), (4427, 2), (2901, 2), (73587, 2), (244, 2), (148, 2), (3928, 2), (31156, 2)

# 2. Evaluation metrics
Implement evaluation metrics for either rating predictions (split metrics) or for top-n recommendations (loo metric, full metric)

In [4]:
def get_hit_rate(anti_testset_top_n, testset):
    """Compute the average hit over the users (loo metric)"""
    hits = 0
    total_users = len(testset)
    for user_id, movie_id, _ in testset:
        if user_id in anti_testset_top_n:
            recommendations = [item_id for (item_id, _) in anti_testset_top_n[user_id]]
            if movie_id in recommendations:
                hits += 1
    return hits / total_users if total_users > 0 else 0


def get_novelty(anti_testset_top_n, item_to_rank, **kwargs):
    """Average popularity rank of recommended items (full metric). Higher = more novel."""
    total_novelty = 0
    total_users = len(anti_testset_top_n)
    for user_id, recommendations in anti_testset_top_n.items():
        user_sum = sum(item_to_rank.get(movie_id, len(item_to_rank)) for movie_id, _ in recommendations)
        total_novelty += user_sum
    return total_novelty / total_users if total_users > 0 else 0.0


def get_ndcg_at_k(anti_testset_top_n, testset, k):
    """Average NDCG@k over users (loo metric).
    With one hidden item per user (LOO): NDCG@k = 1/log2(rank+1) if rank <= k, else 0.
    """
    total = 0.0
    n_users = 0
    for user_id, movie_id, _ in testset:
        if user_id in anti_testset_top_n:
            recs = [item_id for item_id, _ in anti_testset_top_n[user_id]]
            top_k = recs[:k]
            if movie_id in top_k:
                rank = top_k.index(movie_id) + 1
                total += 1.0 / np.log2(rank + 1)
            n_users += 1
    return total / n_users if n_users > 0 else 0.0


def get_precision_at_k(anti_testset_top_n, testset, k):
    """Average Precision@k over users (loo metric).
    With one hidden item per user (LOO): Precision@k = 1/k if item in top-k, else 0.
    """
    total = 0.0
    n_users = 0
    for user_id, movie_id, _ in testset:
        if user_id in anti_testset_top_n:
            recs = [item_id for item_id, _ in anti_testset_top_n[user_id]]
            if movie_id in recs[:k]:
                total += 1.0 / k
            n_users += 1
    return total / n_users if n_users > 0 else 0.0


def get_recall_at_k(anti_testset_top_n, testset, k):
    """Average Recall@k over users (loo metric).
    With one hidden item per user (LOO): Recall@k = 1 if item in top-k, else 0.
    """
    hits = 0
    n_users = 0
    for user_id, movie_id, _ in testset:
        if user_id in anti_testset_top_n:
            recs = [item_id for item_id, _ in anti_testset_top_n[user_id]]
            if movie_id in recs[:k]:
                hits += 1
            n_users += 1
    return hits / n_users if n_users > 0 else 0.0


def get_coverage(anti_testset_top_n, item_to_rank, **kwargs):
    """Catalogue coverage (full metric).
    Coverage = |unique recommended items| / |total catalogue|
    """
    recommended_items = set()
    for recommendations in anti_testset_top_n.values():
        for item_id, _ in recommendations:
            recommended_items.add(item_id)
    total_items = len(item_to_rank)
    return len(recommended_items) / total_items if total_items > 0 else 0.0


def get_novelty_miuf(anti_testset_top_n, n_users, item_freq, **kwargs):
    """Mean Inverse User Frequency novelty (full metric).
    MIUF = (1/|R|) * sum_{i in R} -log2(|U_i| / |U|)
    Higher = recommends less popular items.
    """
    user_novelties = []
    for uid, recs in anti_testset_top_n.items():
        if not recs:
            continue
        miuf_scores = [-np.log2(item_freq.get(iid, 1) / n_users) for iid, _ in recs]
        user_novelties.append(np.mean(miuf_scores))
    return float(np.mean(user_novelties)) if user_novelties else 0.0


def build_genre_vectors(df_items):
    """Build normalized binary genre vectors per item (used for ILD)."""
    all_genres = sorted(set(
        g for genres in df_items[C.GENRES_COL].fillna('').str.split('|')
        for g in genres if g and g != '(no genres listed)'
    ))
    genre_index = {g: i for i, g in enumerate(all_genres)}
    vectors = {}
    for mid, row in df_items.iterrows():
        vec = np.zeros(len(all_genres))
        for g in str(row[C.GENRES_COL]).split('|'):
            if g in genre_index:
                vec[genre_index[g]] = 1.0
        norm = np.linalg.norm(vec)
        vectors[mid] = vec / norm if norm > 0 else vec
    return vectors


def get_diversity_ild(anti_testset_top_n, genre_vectors, **kwargs):
    """Intra-List Diversity (full metric).
    ILD = (1 / |R|(|R|-1)) * sum_{i in R} sum_{j in R} d(i,j)
    d(i,j) = 1 - cos(g_i, g_j) — cosine dissimilarity on normalized genre vectors.
    Diagonal = 0 since d(i,i) = 0, so np.sum covers all i≠j pairs correctly.
    Higher = more diverse recommendations across genres.
    """
    user_ilds = []
    for uid, recs in anti_testset_top_n.items():
        ids = [iid for iid, _ in recs if iid in genre_vectors]
        R = len(ids)
        if R < 2:
            continue
        vecs = np.array([genre_vectors[i] for i in ids])
        sim = vecs @ vecs.T
        total_dist = np.sum(1 - sim)  # diagonal = 1-1 = 0, so this sums i≠j pairs
        user_ilds.append(total_dist / (R * (R - 1)))
    return float(np.mean(user_ilds)) if user_ilds else 0.0

def get_serendipity(anti_testset_top_n, user_history, genre_vectors, **kwargs):
    user_scores = []

    for uid, recs in anti_testset_top_n.items():
        history = user_history.get(uid, [])
        history = [iid for iid in history if iid in genre_vectors]

        if not history:
            continue

        history_vecs = np.array([genre_vectors[iid] for iid in history])
        rec_scores = []

        for iid, pred_score in recs:
            if iid not in genre_vectors:
                continue

            item_vec = genre_vectors[iid]

            similarities = history_vecs @ item_vec
            expectedness = np.max(similarities)
            unexpectedness = 1 - expectedness

            relevance = (pred_score - 0.5) / 4.5
            relevance = max(0, min(1, relevance))

            ser = relevance * unexpectedness
            rec_scores.append(ser)

        if rec_scores:
            user_scores.append(np.mean(rec_scores))

    return float(np.mean(user_scores)) if user_scores else 0.0

## Pitfalls of Using a Sum of Ranks as a Novelty Metric

1. **Sensitivity to N (top-n size)**: If two models recommend different numbers
   of items (e.g. 10 vs 40), the sum of ranks will be mechanically higher for
   the one recommending more items, without being truly more "novel".

2. **Linearity of ranks**: The difference between rank 1 and rank 100 is
   treated the same as between rank 1000 and rank 1100. However, moving from
   the most popular movie to the 100th is a far more radical shift in popularity
   than moving from rank 1000 to 1100. A logarithmic scale would better capture
   this reality.

3. **Quality vs. Novelty trade-off**: A model could achieve a very high novelty
   score by recommending the least popular (and potentially worst) movies on the
   platform that nobody wants to watch. Novelty should always be balanced with
   precision metrics (MAE, RMSE, Hit Rate) to ensure recommendations remain
   relevant.

4. **Dependence on catalogue size**: A rank of 500 in a catalogue of 600 movies
   does not have the same meaning as a rank of 500 in a catalogue of 100,000
   movies. The metric is therefore not comparable across systems with catalogues
   of different sizes.

# 3. Evaluation workflow
Load data, evaluate models and save the experimental outcomes

In [ ]:
from surprise import Dataset, Reader
np.random.seed(1)
rd.seed(1)

AVAILABLE_METRICS = {
    "split": {
        "mae": (accuracy.mae, {'verbose': False}),
        "rmse": (accuracy.rmse, {'verbose': False})
    },
    "loo": {
        "hit_rate": (get_hit_rate, {}),
        "ndcg@5": (get_ndcg_at_k, {"k": 5}),
        "ndcg@10": (get_ndcg_at_k, {"k": 10}),
        "ndcg@20": (get_ndcg_at_k, {"k": 20})
    },
    "full": {
        "coverage": (get_coverage, {}),
        "miuf": (get_novelty_miuf, {}),
        "ild": (get_diversity_ild, {}),
        "serendipity": (get_serendipity, {})
    }
}

df_ratings_pd = load_ratings(surprise_format=False)
debug_users = df_ratings_pd[C.USER_ID_COL].drop_duplicates().sample(
    frac=1,
    random_state=1
)

df_ratings_pd = df_ratings_pd[
    df_ratings_pd[C.USER_ID_COL].isin(debug_users)
]

sp_ratings = Dataset.load_from_df(
    df_ratings_pd[[C.USER_ID_COL, C.ITEM_ID_COL, C.RATING_COL]],
    Reader(rating_scale=(0.5, 5))
)
precomputed_dict = precompute_information(df_ratings_pd)
evaluation_report = create_evaluation_report(EvalConfig(), sp_ratings, precomputed_dict, AVAILABLE_METRICS)
export_evaluation_report(evaluation_report)
display(evaluation_report)

Handling model baseline_1
Training split predictions
- computing metric rmse
- computing metric mae
Training loo predictions
Training full predictions
Handling model baseline_2
Training split predictions
- computing metric rmse
- computing metric mae
Training loo predictions
Training full predictions
Handling model baseline_3
Training split predictions
- computing metric rmse
- computing metric mae
Training loo predictions
Training full predictions
Handling model baseline_4
Training split predictions
- computing metric rmse
- computing metric mae
Training loo predictions
Training full predictions
Handling model ContentBased_ridge_cv
Training split predictions
- computing metric rmse
- computing metric mae
Training loo predictions
Training full predictions


,rmse,mae,hit_rate,ndcg@5,ndcg@10,ndcg@20,coverage,miuf,ild,serendipity
baseline_1,1.732666,1.548323,0.2,0.043068,0.043068,0.043068,0.135283,0.804190,0.707534,0.020954
baseline_2,1.738566,1.418907,0.0,0.000000,0.000000,0.000000,0.227828,2.543399,0.741358,0.047420
baseline_3,0.898397,0.690466,0.2,0.043068,0.043068,0.043068,0.135283,0.804190,0.707534,0.039530
baseline_4,0.824246,0.627766,0.1,0.100000,0.100000,0.100000,0.171594,1.659191,0.699741,0.046016
ContentBased_ridge_cv,0.713832,0.527063,0.0,0.000000,0.000000,0.000000,0.172879,2.151551,0.623401,0.043548


## Evaluation Report Observations

**Note**: Results obtained on the test dataset (6 users, 10 items),
used only to verify that the pipeline is working correctly.

### 1. Precision Metrics (MAE & RMSE)
- **Baseline 4 (SVD)** is the best performing model on MAE (0.954). This is
  consistent since SVD is a learning algorithm that minimizes prediction error,
  unlike the static baselines.
- **Baseline 1** is the least performant (MAE=1.125): always predicting the same
  value captures none of the nuances of user preferences.
- RMSE is systematically higher than MAE for all models, which is expected:
  RMSE penalizes large errors more heavily (squared differences).

### 2. Hit Rate (1.0)
- All 4 models display a Hit Rate of 1.0 (100%), which would be impossible
  in a real scenario.
- This is explained by the dataset size: with only ~10 available movies and
  a top_n_value=40, the model will always include the "hidden" movie in its list.
- The metric is correctly implemented, but will only be discriminant on a larger
  dataset (thousands of movies, with only 40 possible recommendations).

### 3. Novelty (~30.17)
- The value is nearly identical across all baselines.
- For models predicting constant or average values (Baseline 1 and 3), the
  ordering of recommendations depends on item appearance order or tie-breaking.
- On such a small dataset, all models end up recommending every available movie,
  making the average popularity rank mechanically identical for everyone.

## Content-Based Models Evaluation

### 1. Random Sample vs. Random Score
Random Sample (RMSE 1.31) outperforms Random Score (RMSE 1.79). Random Sample draws from the user's own rating distribution, centered around their personal mean (~3.5 on MovieLens), whereas Random Score samples uniformly in [0.5, 5] (mean ~2.75). Predictions from Random Sample are therefore structurally closer to real ratings.

### 2. Linear Regression (Intercept True vs. False)
Linear Regression with `fit_intercept=True` (RMSE 0.93) vastly outperforms `fit_intercept=False` (RMSE 1.55). Without an intercept, the model must explain a ~3.5-star average using only `coef × title_length`, which is impossible without distorting the coefficient. The intercept captures the user's baseline rating tendency — essentially their mean. With a single weak feature like title length, the intercept does almost all the work: 0.93 is roughly what a "predict the user's mean" baseline would achieve.

### 3. Comparing more advanced models
As base features, we used `all_content_tmdb_tags2000`, which concatenates genome scores (1128 dims), a rich TF-IDF on user tags (up to 2000 features, bigrams, sublinear TF), normalized release year + decade one-hot, genres (TF-IDF), and TMDB metadata (runtime, language, country, studio, directors, cast, keywords, collection, budget, writers, release date, overview TF-IDF). The total feature space spans several thousand dimensions. We compared two regressors:
- **Ridge**: L2-regularized linear regression with a fixed `alpha=1.0` for all users.
- **RidgeCV**: same model, but `alpha` is selected per user via cross-validation over `1e-4` to `1e5`.

**RidgeCV (0.74) outperforms standard Ridge (0.93)** because with several thousand features and only tens to hundreds of ratings per user, `alpha=1.0` is severely under-regularized: Ridge overfits and falls back to roughly the same RMSE as a "predict the user's mean" model. RidgeCV picks larger alphas for sparse profiles (strong shrinkage) and smaller ones for rich profiles, adapting regularization to each user's data volume. Notably, **RidgeCV (0.744) even beats the best collaborative baseline so far, SVD (0.817)**, using only item content and per-user ridge profiles.